# Perform an inventory audit of an Excel HTA model

This is attempting to recreate the first stage of the Microsoft CoPilot prompt i.e. identifying all inputs. This includes inputs that are hardcoded into formulas.

The diagram below illustrates the general approach (subject to change):

<img src='images/model_qc_workflow.png' width='800' height='800' />





## 1. Imports

### 1.1. General imports

In [1]:
import sys
from pathlib import Path
from rich import print as printr
import json
import pandas as pd

import warnings
# to ignore a openpyxl UserWarning
warnings.filterwarnings('ignore')  # Suppress all warnings

### 1.2 `xl-ray`

In [2]:
from xl_ray.extractor import (
    load_workbooks, 
    extract_metadata, 
    extract_vba_modules, 
    extract_named_ranges,
    extract_worksheets 
)

from xl_ray.schema import ExcelModelData

from xl_ray.audit import (
    detect_magic_numbers, # TM note - I wonder if this is useful for LLM hint? 

)

from xl_ray.audit import 

### 1.3 LLM workflow imports

In [3]:
# standard langchain 
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# custom in input_selection module
from input_selection import (
    SELECTION_INSTRUCTIONS,
    SheetSelection,
    build_selection_payload,
    selected_sheet_names,
    validate_selection,
    build_selection_text
)

## 2. Import model via `xl-ray`

In [4]:
TEST_MODEL = "models/Exeter_model_template_Mar25.xlsm"

In [5]:
test_file = Path(TEST_MODEL)

if not test_file.exists():
    print(f"Error: Could not find {test_file}")
    sys.exit(1)

print(f"Loading '{test_file.name}'...")
wb_formulas, wb_values = load_workbooks(test_file)

print("Extracting VBA modules...")
vba_modules = extract_vba_modules(test_file, wb_formulas)

print("Extracting Named Ranges...")
named_ranges = extract_named_ranges(wb_formulas)

print("Extracting Worksheets, Cells, and Tables (this may take a moment)...")
worksheets = extract_worksheets(wb_formulas, wb_values)

metadata = extract_metadata(
    path=Path(test_file),
    wb_formulas=wb_formulas, 
    vba_modules=vba_modules, 
    named_ranges=named_ranges
)

metadata.has_data_tables = any(ws.tables for ws in worksheets.values())
metadata.has_array_formulas = any(
    cell.is_array_formula for ws in worksheets.values() for cell in ws.cells.values()
)

excel_model = ExcelModelData(
    metadata=metadata,
    worksheets=worksheets,
    named_ranges=named_ranges,
    vba_modules=vba_modules
)

print("Import of Excel model complete.")

Loading 'Exeter_model_template_Mar25.xlsm'...
Extracting VBA modules...
Extracting Named Ranges...
Extracting Worksheets, Cells, and Tables (this may take a moment)...
Import of Excel model complete.


## 3. LLM functions

In [6]:
def select_worksheets_using_LLM(
    payload: str,
    system_prompt: str,
    base_url: str = "http://127.0.0.1:8001/v1",
    selected_model: str = "google/gemma-4-26B-A4B-it",
    random_seed: int | None = None,
    temperature: float = 1.0,
) -> SheetSelection:
    """
    Select worksheets for detailed input identification.

    `prompt` should contain the compact workbook summary.

    Returns a validated SheetSelection, not an AIMessage.
    """
    model = ChatOpenAI(
        base_url=base_url,
        api_key="EMPTY",
        model=selected_model,
        temperature=temperature,
        **({"seed": random_seed} if random_seed is not None else {}),
    )

    # used to get structured output
    structured_model = model.with_structured_output(
        SheetSelection,
        method="json_schema",
        include_raw=False,
    )

    selection = structured_model.invoke(
        [
            SystemMessage(content=system_prompt.strip()),
            HumanMessage(content=payload),
        ]
    )

    return selection

## 4. Script to select sheets to include in audit

### 4.1. Build the worksheet selection prompt

In [7]:
print("Building prompt")
payload = build_selection_payload(
    excel_model,
    max_rows_per_sheet = 12,
    max_cells_per_row = 10,
)

# convert JSON to plain text to reduce charactor count
workbook_summary_as_text= build_selection_text(payload)

print("Workbook summary characters:", len(workbook_summary_as_text))

Building prompt
Workbook summary characters: 71907


### 4.2 Prompt LLM to select worksheets

In [8]:
# These are the LLMs currently running.
ENDPOINTS = {
    "qwen": {
        "model": "Qwen/Qwen3.5-35B-A3B",
        "url": "http://127.0.0.1:8000/v1"
    },
    "gemma4": {
        "model": "google/gemma-4-26B-A4B-it", 
        "url": "http://127.0.0.1:8001/v1"
    }
}

In [9]:
%%time

selected_endpoint = "qwen"

print(
    f"{ENDPOINTS[selected_endpoint]["model"]} is selecting sheets "
    "to include in audit"
)

# run the prompt
selection = select_worksheets_using_LLM(
    payload=workbook_summary_as_text,
    system_prompt = SELECTION_INSTRUCTIONS,
    base_url=ENDPOINTS[selected_endpoint]["url"],
    selected_model=ENDPOINTS[selected_endpoint]["model"],
    random_seed=42,
)

Qwen/Qwen3.5-35B-A3B is selecting sheets to include in audit
CPU times: user 351 ms, sys: 22.9 ms, total: 374 ms
Wall time: 23.4 s


In [10]:
type(selection)

input_selection.SheetSelection

### 4.3. Save results and convert to `DataFrame` 

In [11]:
# Validate worksheet names and coverage.
selection = validate_selection(
    selection.model_dump_json(),
    excel_model,
)

sheets_to_audit = selected_sheet_names(
    selection,
    include_review=True,
)

# Convert the validated Pydantic objects to a DataFrame.
selection_df = pd.DataFrame(
    [
        {
            "sheet": item.sheet,
            "decision": item.decision,
            "role": item.likely_role,
            "reason": item.reason,
            "evidence": "\n".join(item.evidence),
            "uncertainty": item.uncertainty,
        }
        for item in selection.sheets
    ],
    columns=[
        "sheet",
        "decision",
        "role",
        "reason",
        "evidence",
        "uncertainty",
    ],
)
 
selection_df["selected_for_audit"] = selection_df["sheet"].isin(
    sheets_to_audit
)

# Retain the worksheet objects for the next LLM stage.
selected_worksheets = {
    name: excel_model.worksheets[name]
    for name in sheets_to_audit
}

In [12]:
# show sheets excluded from audit
selection_df[selection_df["selected_for_audit"] == False]

,sheet,decision,role,reason,evidence,uncertainty,selected_for_audit
0,Cover,skip,unknown,"Worksheet contains metadata only (client name,...",Row 5: C5='[Client name]'\nRow 10: C10='PenTAG...,,False
1,Background,skip,unknown,Worksheet contains descriptive text and strate...,Row 6: C6='Model diagram '\nRow 34: C34='Cells...,,False
18,Model QC,skip,unknown,Contains checklist for quality control of the ...,Row 1: A1='White box tests' | B1='Test no.' | ...,,False
19,Navigation,skip,unknown,Contains navigation links and layout notes only.,Row 1: C1='Navigation'\nRow 10: E10='Backgroun...,,False
24,References,skip,unknown,Contains bibliographic citations and sources f...,Row 4: E4=1 | F4='PSSRU 2023' | G4='PSSRU 2023...,,False


In [13]:
# show sheets included in stage 2
selection_df[selection_df["selected_for_audit"] == True]

,sheet,decision,role,reason,evidence,uncertainty,selected_for_audit
2,Model_settings,inspect,inputs,Contains user-specified model settings includi...,Row 18: C18='Life Years discount rate' | D18=0...,,True
3,Treatment_costs,inspect,inputs,Contains treatment cost inputs including unit ...,Row 7: C7=2 | D7='Treatment 1' | E7=100 | G7=2...,,True
4,HCRU,inspect,inputs,Contains health care resource use unit costs a...,Row 7: C7=2 | D7='Resource 2' | E7=20 | F7=201...,,True
5,Societal_costs,inspect,inputs,Contains societal resource use unit costs and ...,Row 16: C16=2 | D16='Well' | E16='Presenteeism...,,True
6,Inflation,inspect,inputs,Contains historical inflation rates by year so...,Row 5: D5='2008/2009' | E5=2009 | G5=0.027 | H...,,True
7,Effectiveness,inspect,inputs,Contains treatment effectiveness inputs includ...,Row 11: D11='Intervention 2 vs No treatment' |...,,True
8,LifeTable,review,inputs,Large mortality table containing life expectan...,Row 87: C87=73 | F87=[formula] =1-EXP(-1*(-LN(...,Sampled rows are heavily formula-driven; uncer...,True
9,Adverse_events,inspect,inputs,"Contains adverse event costs, HRQoL decrements...",Row 5: C5=1 | D5='Event 1' | E5=100 | G5=[form...,,True
10,HRQoL,inspect,inputs,"Contains utility values, regression parameters...",Row 8: D8='Baseline utility' | E8=0.8\nRow 14:...,,True
11,Parameters,inspect,inputs,Centralized parameter table linking all inputs...,Row 13: D13='Model_settings' | G13='pop_settin...,,True


In [14]:
# Take from the copilot oneshot example. 
# Unique list of sheets that contain flagged inputs.
copilot_included = [
    "Model_settings",
    "Treatment_costs",
    "HCRU",
    "Societal_costs",
    "Effectiveness",
    "Adverse_events",
    "HRQoL",
    "LifeTable",
    "Parameters",
    "PSA",
    "OWSA",
    "Scenario_analysis",
    "Engine(1)",
    "Engine(2)",
    "Engine(3)",
]

len(copilot_included)

15

## 5. What about xl-ray magic numbers?

I don't need to use the LLM for finding all cells with hard coded values when I have `xl-ray`

The function `detect_magic_numbers` captures all formulas with hard coded values.  Returned as a list

**To consider:**

* Do I just need to organise these and use the LLM to provide the context and understanding?
* What about just cells that are hard coded?
* Can `xl-ray` provide simple functionality to help the LLM on where parameters might be stored?
* How hard would it be to identifier potential ICER cells with `xl-ray`

**Maybe it doesn't matter at this stage?**



In [28]:
from IPython.display import display, Markdown

# returns a list
cells_with_hard_coded_settings = detect_magic_numbers(excel_model)

# peek
Markdown("**Preview of magic numbers**\n")
cells_with_hard_coded_settings[:3]

[{'sheet': 'Model_settings',
  'cell': 'D16',
  'formula': '=ctrl_setting_CycleLength*7/365.25',
  'magic_numbers_found': ['7', '365.25']},
 {'sheet': 'HCRU',
  'cell': 'H15',
  'formula': '=(INDEX(p_r_HCRU_HealthStateResourceUse,C15)/365.25)*ctrl_setting_CycleLength*7',
  'magic_numbers_found': ['365.25', '7']},
 {'sheet': 'HCRU',
  'cell': 'H16',
  'formula': '=(INDEX(p_r_HCRU_HealthStateResourceUse,C16)/365.25)*ctrl_setting_CycleLength*7',
  'magic_numbers_found': ['365.25', '7']}]

## End of Selection of Sheets.  